In [1]:
install.packages("hoopR")


The downloaded binary packages are in
	/var/folders/sz/k_920m494wd37gp8s3kxnlxw0000gn/T//RtmpxOi59i/downloaded_packages


In [7]:
#!/usr/bin/env Rscript

suppressPackageStartupMessages({
  library(hoopR)
  library(dplyr)
  library(readr)
  library(stringr)
  library(purrr)
})

SEASONS <- 2000:2000  # season END years (2009-10 .. 2017-18)
BASE_OUT_DIR <- file.path("..", "data", "games_live2")
dir.create(BASE_OUT_DIR, recursive = TRUE, showWarnings = FALSE)

sanitize_team <- function(x) {
  x <- as.character(x)
  x <- toupper(str_trim(x))
  x <- gsub("[^A-Z0-9]+", "_", x)
  x <- gsub("_+", "_", x)
  x <- gsub("^_|_$", "", x)
  ifelse(is.na(x) | x == "", "UNK", x)
}

load_schedule_safely <- function(season_end_year) {
  candidates <- list(
    function() hoopR::load_nba_schedule(seasons = season_end_year),
    function() hoopR::nba_schedule(seasons = season_end_year)   # some versions differ
  )
  for (fn in candidates) {
    out <- try(fn(), silent = TRUE)
    if (!inherits(out, "try-error") && is.data.frame(out) && nrow(out) > 0) return(out)
  }
  stop(sprintf("Could not load NBA schedule metadata via hoopR for season ending %d.", season_end_year))
}

pick_col <- function(df, candidates) {
  cand <- candidates[candidates %in% names(df)]
  if (length(cand) == 0) return(NA_character_)
  cand[[1]]
}

process_one_season <- function(season_end_year) {
  season_dir <- file.path(BASE_OUT_DIR, as.character(season_end_year))
  dir.create(season_dir, recursive = TRUE, showWarnings = FALSE)

  message(sprintf("\n==============================\nSeason ending %d\n==============================", season_end_year))
  message(sprintf("Loading PBP for season ending %d ...", season_end_year))

  pbp <- hoopR::load_nba_pbp(seasons = season_end_year) %>%
    mutate(game_id = as.character(game_id))

  if (!"game_id" %in% names(pbp)) stop("PBP is missing `game_id`.")

  game_ids <- unique(pbp$game_id)
  message(sprintf("PBP rows: %d | games: %d", nrow(pbp), length(game_ids)))

  message("Loading schedule metadata to map game_id -> teams ...")
  sched <- load_schedule_safely(season_end_year)

  nm <- names(sched)

  gid_col <- c("game_id", "gameId", "id", "espn_game_id", "gameID")[
    c("game_id", "gameId", "id", "espn_game_id", "gameID") %in% nm
  ][1]
  if (is.na(gid_col)) stop("Schedule metadata did not contain a recognizable game id column.")

  sched <- sched %>% mutate(game_id = as.character(.data[[gid_col]]))

  away_col <- pick_col(sched, c("away_team_abbreviation","away_abbreviation","away_team","awayTeamAbbreviation","away_team_abb","away_abbr"))
  home_col <- pick_col(sched, c("home_team_abbreviation","home_abbreviation","home_team","homeTeamAbbreviation","home_team_abb","home_abbr"))

  away_name_col <- pick_col(sched, c("away_team_name","awayTeamName","away_name","awayTeam","away_team_full"))
  home_name_col <- pick_col(sched, c("home_team_name","homeTeamName","home_name","homeTeam","home_team_full"))

  games_meta <- sched %>%
    transmute(
      game_id,
      away = if (!is.na(away_col)) .data[[away_col]] else if (!is.na(away_name_col)) .data[[away_name_col]] else NA_character_,
      home = if (!is.na(home_col)) .data[[home_col]] else if (!is.na(home_name_col)) .data[[home_name_col]] else NA_character_
    ) %>%
    mutate(
      away = sanitize_team(away),
      home = sanitize_team(home)
    ) %>%
    distinct(game_id, .keep_all = TRUE)

  pbp2 <- pbp %>% left_join(games_meta, by = "game_id")

  missing_team_ids <- pbp2 %>%
    distinct(game_id, away, home) %>%
    filter(away == "UNK" | home == "UNK")

  if (nrow(missing_team_ids) > 0) {
    message(sprintf("Warning: %d games missing team info (UNK).", nrow(missing_team_ids)))
  }

  message(sprintf("Writing per-game CSVs to %s ...", season_dir))

  written <- 0L
  pbp2 %>%
    group_by(game_id, away, home) %>%
    group_walk(function(df, key) {
      gid  <- as.character(key$game_id[[1]])
      away <- as.character(key$away[[1]])
      home <- as.character(key$home[[1]])

      filename <- sprintf("%s_%s_%s.csv", gid, away, home)
      path <- file.path(season_dir, filename)

      write_csv(df, path, na = "")
      written <<- written + 1L
      if (written %% 50L == 0L) message(sprintf("...written %d", written))
    })

  message(sprintf("Done season %d. Wrote %d files.", season_end_year, written))
  invisible(written)
}

# -----------------------------
# Loop through seasons 2010-2018
# -----------------------------
total_written <- 0L

for (season_end_year in SEASONS) {
  wrote <- tryCatch(
    process_one_season(season_end_year),
    error = function(e) {
      message(sprintf("ERROR in season %d: %s", season_end_year, conditionMessage(e)))
      0L
    }
  )
  total_written <- total_written + wrote
}

message(sprintf("\nAll done. Total files written across seasons: %d", total_written))


Season ending 2000

Loading PBP for season ending 2000 ...

ERROR in season 2000: seasons >= 2002 is not TRUE


All done. Total files written across seasons: 0



In [8]:
seasons <- 2000:2001

for (season in seasons) {
  cat("Fetching season", season, "...\n")
  
  tryCatch({
    schedule <- load_nba_schedule(seasons = season)
    
    # Build filename: 2003 -> "schedule_2002-03.csv", 2010 -> "schedule_2009-10.csv"
    start_year <- season - 1
    end_yy <- sprintf("%02d", season %% 100)
    filename <- paste0("schedule_", start_year, "-", end_yy, ".csv")
    
    write.csv(schedule, file.path("../data/schedules2", filename), row.names = FALSE)
    cat("  Saved:", filename, "-", nrow(schedule), "rows\n")
  }, error = function(e) {
    cat("  ERROR for season", season, ":", conditionMessage(e), "\n")
  })
  
  Sys.sleep(1)  # be polite to the API
}

cat("Done!\n")

Fetching season 2000 ...
  ERROR for season 2000 : seasons >= 2002 is not TRUE 
Fetching season 2001 ...
  ERROR for season 2001 : seasons >= 2002 is not TRUE 
Done!
